In [12]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import json
import ast
from pathlib import Path

def parse_xml_to_df(xml_path, entity_tag='book'):
    """Parses XML files into a DataFrame handling scalar and list elements."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    records = []
    
    for entity in root.findall(f'.//{entity_tag}'):
        data = {}
        for child in entity:
            if len(child) > 0:
                vals = [sub.text for sub in child if sub.text]
                data[child.tag] = vals if vals else None
            else:
                data[child.tag] = child.text if child.text else None
        records.append(data)
    return pd.DataFrame(records)

def load_csv_with_json_lists(csv_path):
    """Loads a CSV and safely parses stringified lists or JSON arrays without crashing on plain text."""
    df = pd.read_csv(csv_path)
    
    def parse_list_string(val):
        if isinstance(val, str) and val.strip().startswith('['):
            try:
                # Versuch 1: Standard-JSON (z. B. ["Author 1", "Author 2"])
                return json.loads(val)
            except (json.JSONDecodeError, TypeError):
                try:
                    # Versuch 2: Python-Listen-Syntax (z. B. ['Author 1', 'Author 2'])
                    res = ast.literal_eval(val)
                    if isinstance(res, list):
                        return res
                except (ValueError, SyntaxError):
                    pass
        return val

    for col in df.columns:
        # Prüfen, ob die Spalte mindestens einen Eintrag hat, der wie eine Liste aussieht
        sample = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else None
        if isinstance(sample, str) and sample.strip().startswith('['):
            df[col] = df[col].apply(parse_list_string)
            
    return df

def clean_missing_values(df):
    """Replaces empty strings, empty lists, and 'None' strings with np.nan safely."""
    def is_empty(x):
        if x is None:
            return True
        if isinstance(x, (list, np.ndarray, tuple)):
            return len(x) == 0
        if pd.isna(x):
            return True
        if isinstance(x, str):
            return x.strip() in ["", "[]", "None", "nan", "NaN"]
        return False

    return df.map(lambda x: np.nan if is_empty(x) else x)

# ==========================================
# LOAD DATASETS
# ==========================================
dataset_paths = {
    'DBpedia': ('data/dbpedia_book_data.xml', 'xml'),
    'Goodreads': ('data/books_1.Best_Books_Ever.csv', 'csv'),
    'Amazon': ('data/amazon_books_subset.csv', 'csv')
}

datasets = {}

# DBpedia
if Path(dataset_paths['DBpedia'][0]).exists():
    datasets['DBpedia'] = parse_xml_to_df(dataset_paths['DBpedia'][0])

# Goodreads
if Path(dataset_paths['Goodreads'][0]).exists():
    datasets['Goodreads'] = load_csv_with_json_lists(dataset_paths['Goodreads'][0])

# Amazon
if Path(dataset_paths['Amazon'][0]).exists():
    datasets['Amazon'] = load_csv_with_json_lists(dataset_paths['Amazon'][0])

In [13]:
# ==========================================
# MISSING VALUES ANALYSIS (>30% THRESHOLD)
# ==========================================
missing_reports = []

for name, df in datasets.items():
    df_cleaned = clean_missing_values(df)
    missing_pct = (df_cleaned.isnull().mean() * 100).round(2)
    missing_counts = df_cleaned.isnull().sum()
    total_records = len(df_cleaned)
    
    for col in df_cleaned.columns:
        pct = missing_pct[col]
        missing_reports.append({
            'Dataset': name,
            'Attribute Name': col,
            'Missing Count': missing_counts[col],
            'Total Records': total_records,
            'Missing (%)': pct,
            'High Missing (>30%)': pct > 30.0
        })

df_missing_summary = pd.DataFrame(missing_reports)

# Display attributes with >30% missing values
high_missing_df = df_missing_summary[df_missing_summary['High Missing (>30%)']].sort_values(by=['Dataset', 'Missing (%)'], ascending=[True, False])

print("=== ATTRIBUTES WITH > 30% MISSING VALUES ===")
if not high_missing_df.empty:
    display(high_missing_df[['Dataset', 'Attribute Name', 'Missing Count', 'Total Records', 'Missing (%)']])
else:
    print("No attributes found with more than 30% missing values.")

=== ATTRIBUTES WITH > 30% MISSING VALUES ===


,Dataset,Attribute Name,Missing Count,Total Records,Missing (%)
2,DBpedia,year,18567,20981,88.49
4,DBpedia,language,18378,20981,87.59
7,DBpedia,genres,10303,20981,49.11
18,Goodreads,edition,47523,52478,90.56
23,Goodreads,awards,41864,52478,79.77
27,Goodreads,setting,40900,52478,77.94
16,Goodreads,characters,38712,52478,73.77
9,Goodreads,series,29009,52478,55.28
22,Goodreads,firstPublishDate,21326,52478,40.64


In [14]:
# ==========================================
# TABLE 1: DATASET PROFILE & ATTRIBUTE LIST WITH (MV)
# ==========================================
table_1_rows = []

# Identifikatoren, die nicht als Daten-Attribute in Table 1 gezählt werden sollen
IGNORE_ATTRIBUTES = {'amazon_id', 'asin', 'isbn10', 'isbn13'}

for name, df in datasets.items():
    df_cleaned = clean_missing_values(df)
    missing_pct = df_cleaned.isnull().mean() * 100
    
    # Filtere reine Identifikatoren für die Attributzählung heraus
    attr_cols = [col for col in df_cleaned.columns if col not in IGNORE_ATTRIBUTES]
    
    # Format attribute list and mark >30% missing values with (MV)
    formatted_attrs = []
    for col in attr_cols:
        if missing_pct[col] > 30:
            formatted_attrs.append(f"{col} (MV)")
        else:
            formatted_attrs.append(col)
            
    filepath, fileformat = dataset_paths.get(name, ('unknown', 'csv'))
    
    table_1_rows.append({
        'Dataset': name,
        'Source(*)': filepath,
        'Format': fileformat,
        '# of entities': len(df_cleaned),
        '# of attributes': len(attr_cols),
        'List of attributes (**)': ", ".join(formatted_attrs)
    })

df_table_1 = pd.DataFrame(table_1_rows)
print("=== TABLE 1: SCHEMA AND BASIC PROFILE OF EACH DATASET ===")
display(df_table_1)

print("\n=== ATTRIBUTE LISTS PER DATASET ===")
for _, row in df_table_1.iterrows():
    print(f"\n[{row['Dataset']}] ({row['# of attributes']} attributes):")
    print(f"  {row['List of attributes (**)']}")



=== TABLE 1: SCHEMA AND BASIC PROFILE OF EACH DATASET ===


,Dataset,Source(*),Format,# of entities,# of attributes,List of attributes (**)
0,DBpedia,data/dbpedia_book_data.xml,xml,20981,8,"title, publisher, year (MV), pages, language (..."
1,Goodreads,data/books_1.Best_Books_Ever.csv,csv,52478,23,"title, series (MV), author, rating, descriptio..."
2,Amazon,data/amazon_books_subset.csv,csv,40000,11,"title, subtitle, authors, publisher, publicati..."



=== ATTRIBUTE LISTS PER DATASET ===

[DBpedia] (8 attributes):
  title, publisher, year (MV), pages, language (MV), isbns, authors, genres (MV)

[Goodreads] (23 attributes):
  title, series (MV), author, rating, description, language, isbn, genres, characters (MV), bookFormat, edition (MV), pages, publisher, publishDate, firstPublishDate (MV), awards (MV), numRatings, ratingsByStars, likedPercent, setting (MV), bbeScore, bbeVotes, price

[Amazon] (11 attributes):
  title, subtitle, authors, publisher, publication_date, language, pages, genres, average_rating, ratings_count, price


In [15]:
# ==========================================
# 1. DYNAMIC ATTRIBUTE TYPE INFERENCE
# ==========================================
def infer_attribute_type(series):
    """
    Infers semantic attribute type (string, string/list, numeric, date) 
    based on the actual data content of a series.
    """
    valid_vals = series.dropna()
    if len(valid_vals) == 0:
        return 'string'
    
    sample = valid_vals.iloc[0]
    
    # Check for List types
    if isinstance(sample, (list, tuple, np.ndarray)):
        return 'string/list'
    if isinstance(sample, str) and sample.strip().startswith('['):
        return 'string/list'
    
    # Check for Numeric types
    if pd.api.types.is_numeric_dtype(series):
        return 'numeric'
    
    # Check for Date / Year types
    col_name = series.name.lower()
    if 'date' in col_name or 'year' in col_name:
        return 'date'
        
    return 'string'

# ==========================================
# 2. DYNAMIC TABLE 2 GENERATION
# ==========================================
all_columns = set()
for df in datasets.values():
    all_columns.update(df.columns)

table_2_rows = []

for col in sorted(list(all_columns)):
    # Identify which datasets contain this attribute
    found_in = [ds_name for ds_name, df in datasets.items() if col in df.columns]
    
    # Get a sample dataframe that contains this column to infer its type
    sample_df = next(df for df in datasets.values() if col in df.columns)
    attr_type = infer_attribute_type(sample_df[col])
    
    table_2_rows.append({
        'Attribute name': col,
        'Attribute type': attr_type,
        'Datasets in which found': ", ".join(found_in),
        'Count Datasets': len(found_in)
    })

df_table_2 = pd.DataFrame(table_2_rows)

# Display Full Table 2
print("=== TABLE 2: INTEGRATED SCHEMA OVERLAP (ALL ATTRIBUTES) ===")
display(df_table_2[['Attribute name', 'Attribute type', 'Datasets in which found']])

# ==========================================
# 3. OVERLAPPING ATTRIBUTES ONLY (COMPACT TABLE)
# ==========================================
# Filter attributes that appear in 2 or more datasets
df_overlapping = df_table_2[df_table_2['Count Datasets'] >= 2].drop(columns=['Count Datasets'])

print("\n=== OVERLAPPING ATTRIBUTES ONLY (FOUND IN AT LEAST 2 DATASETS) ===")
if not df_overlapping.empty:
    display(df_overlapping)
else:
    print("No exact overlapping attribute names found between the datasets.")

# ==========================================
# 4. ENTITY OVERLAP CHECK (DYNAMIC PAIRWISE MATCH)
# ==========================================
clean_str = lambda s: str(s).lower().strip() if pd.notna(s) else ""

def get_first_author(val):
    if isinstance(val, (list, tuple, np.ndarray)) and len(val) > 0:
        return clean_str(val[0])
    return clean_str(val)

def extract_author_col(df):
    """Finds author/authors/author_name column dynamically."""
    for col in ['authors', 'author', 'author_name']:
        if col in df.columns:
            return df[col]
    return pd.Series([""] * len(df))

print("\n=== ENTITY OVERLAP CHECK ===")
ds_names = list(datasets.keys())

for i in range(len(ds_names)):
    for j in range(i + 1, len(ds_names)):
        name1, name2 = ds_names[i], ds_names[j]
        df1, df2 = datasets[name1], datasets[name2]
        
        # Title Exact Match
        titles1 = set(df1['title'].apply(clean_str)) - {""}
        titles2 = set(df2['title'].apply(clean_str)) - {""}
        overlap_titles = titles1.intersection(titles2)
        
        # Title + First Author Match
        key1 = df1['title'].apply(clean_str) + " | " + extract_author_col(df1).apply(get_first_author)
        key2 = df2['title'].apply(clean_str) + " | " + extract_author_col(df2).apply(get_first_author)
        
        keys1 = set(key1[df1['title'].notnull()]) - {" | "}
        keys2 = set(key2[df2['title'].notnull()]) - {" | "}
        overlap_keys = keys1.intersection(keys2)
        
        print(f"Overlap {name1} <-> {name2}:")
        print(f"  - By Title (Exact):           {len(overlap_titles)} books")
        print(f"  - By Title + First Author:    {len(overlap_keys)} books")

=== TABLE 2: INTEGRATED SCHEMA OVERLAP (ALL ATTRIBUTES) ===


,Attribute name,Attribute type,Datasets in which found
0,amazon_id,numeric,Amazon
1,asin,string,Amazon
2,author,string,Goodreads
3,authors,string/list,"DBpedia, Amazon"
4,average_rating,numeric,Amazon
5,awards,string/list,Goodreads
6,bbeScore,numeric,Goodreads
7,bbeVotes,numeric,Goodreads
8,bookFormat,string,Goodreads
9,characters,string/list,Goodreads



=== OVERLAPPING ATTRIBUTES ONLY (FOUND IN AT LEAST 2 DATASETS) ===


,Attribute name,Attribute type,Datasets in which found
3,authors,string/list,"DBpedia, Amazon"
13,genres,string/list,"DBpedia, Goodreads, Amazon"
18,language,string,"DBpedia, Goodreads, Amazon"
21,pages,string,"DBpedia, Goodreads, Amazon"
22,price,string,"Goodreads, Amazon"
25,publisher,string,"DBpedia, Goodreads, Amazon"
32,title,string,"DBpedia, Goodreads, Amazon"



=== ENTITY OVERLAP CHECK ===
Overlap DBpedia <-> Goodreads:
  - By Title (Exact):           2857 books
  - By Title + First Author:    1208 books
Overlap DBpedia <-> Amazon:
  - By Title (Exact):           653 books
  - By Title + First Author:    502 books
Overlap Goodreads <-> Amazon:
  - By Title (Exact):           3401 books
  - By Title + First Author:    1007 books
